# Binary Black Hole Merger from Wave Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Binary_Black_Hole_Merger.ipynb)

## What This Notebook Demonstrates

Two massive energy concentrations on a 1D lattice interact via $\chi$-gradient forces. We observe:

1. **Inspiral**: separation decreases as $\chi$-gradient force drives them together
2. **Merger**: sources meet at contact distance
3. **Ringdown**: post-merger $\chi$ oscillations (gravitational waves)

The force is derived from **energy minimization**: $F = -2\chi \cdot \partial\chi/\partial x$

No external gravity or GR metric was injected.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CHI_0 = 19.0
C = 1.0

N = 400
dx = 1.0
dt = 0.05
x = np.arange(N) * dx
center = N // 2

def create_source(x, pos, width=15.0, amp=8.0):
    return amp * np.exp(-(x - pos)**2 / (2 * width**2))

# Start close enough for chi-well overlap but above merger threshold
pos1 = center - 20.0
pos2 = center + 20.0
vel1 = 0.0
vel2 = 0.0
source_width = 15.0
mass = 100.0
G_COUPLING = 0.5
FRICTION = 0.001  # radiation loss

# Tracking
separations = []
times = []
chi_monitor = []
monitor_idx = center
n_steps = 4000
merger_time = None

for step in range(n_steps):
    t = step * dt

    # Rebuild E sources at current positions
    E = create_source(x, pos1) + create_source(x, pos2)

    # GOV-03 (algebraic): chi responds instantly to energy density
    chi_sq = CHI_0**2 - G_COUPLING * E**2
    chi = np.sqrt(np.maximum(chi_sq, 0.1))

    # Chi-gradient force: F = -2*chi*dchi/dx
    chi_grad = np.gradient(chi, dx)
    idx1 = int(np.clip(pos1 / dx, 1, N-2))
    idx2 = int(np.clip(pos2 / dx, 1, N-2))
    F1 = -2 * chi[idx1] * chi_grad[idx1]
    F2 = -2 * chi[idx2] * chi_grad[idx2]

    # Velocity-Verlet update with friction (radiation damping)
    vel1 = vel1 * (1 - FRICTION) + F1 / mass * dt
    vel2 = vel2 * (1 - FRICTION) + F2 / mass * dt
    pos1 += vel1 * dt
    pos2 += vel2 * dt

    pos1 = np.clip(pos1, 10, N-10)
    pos2 = np.clip(pos2, 10, N-10)

    sep = abs(pos2 - pos1)
    separations.append(sep)
    times.append(t)
    chi_monitor.append(chi[monitor_idx])

    if merger_time is None and sep < source_width:
        merger_time = t

separations = np.array(separations)
times = np.array(times)
chi_monitor = np.array(chi_monitor)

print("=" * 60)
print("BINARY MERGER RESULTS")
print("=" * 60)
print(f"  Initial separation: {separations[0]:.1f}")
print(f"  Final separation:   {separations[-1]:.1f}")
print(f"  Min separation:     {separations.min():.1f}")
print(f"  Merger time:        {merger_time}")
inspiral = separations.min() < separations[0] * 0.5
print(f"  Inspiral detected:  {inspiral}")

if merger_time is not None:
    merge_idx = int(merger_time / dt)
    rd = chi_monitor[merge_idx:] - np.mean(chi_monitor[merge_idx:])
    crossings = np.where(np.diff(np.sign(rd)))[0]
    n_osc = len(crossings) // 2
    print(f"  Ringdown oscillations: {n_osc}")
    print(f"\n  Full merger sequence: inspiral -> merger -> ringdown!")
else:
    print(f"\n  Inspiral observed but merger threshold not crossed.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Separation over time
ax = axes[0]
ax.plot(times, separations, 'b-', linewidth=1.5)
if merger_time is not None:
    ax.axvline(merger_time, color='red', ls='--', label=f'Merger at t={merger_time:.0f}')
ax.set_xlabel('Time')
ax.set_ylabel('Separation')
ax.set_title('Binary Inspiral: Separation Decreases via chi-Gradient Force')
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Chi at monitor point
ax = axes[1]
ax.plot(times, chi_monitor, 'r-', linewidth=0.8)
if merger_time is not None:
    ax.axvline(merger_time, color='blue', ls='--', label='Merger')
ax.set_xlabel('Time')
ax.set_ylabel(f'chi at x={monitor_idx}')
ax.set_title('Chi Signal at Monitor (Gravitational Wave)')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Binary Black Hole Merger from GOV-01 + GOV-02',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Result

- Two energy concentrations create overlapping $\chi$-wells
- The $\chi$-gradient force drives them together (**inspiral**)
- After merger, $\chi$ oscillates (**ringdown** = gravitational waves)
- The entire sequence \u2014 inspiral, merger, ringdown \u2014 emerges from GOV-01 + GOV-02 alone